In [0]:
api_bronze = spark.table(
    "ingestion_autoloader.ingestion_demo.api_bronze"
)


In [0]:
api_bronze.printSchema()


In [0]:
from pyspark.sql.functions import col, trim, lower

api_silver = (
    api_bronze
    .select(
        col("id").cast("long").alias("customer_id"),
        trim(col("name")).alias("customer_name"),
        lower(trim(col("email"))).alias("email"),
        trim(col("address")).alias("city"),
        trim(col("company.name")).alias("company_name"),
        col("phone"),
        col("website"),
        col("_ingestion_timestamp"),
        col("_source")
    )
)
display(api_silver)


In [0]:
api_silver = api_silver.filter(
    col("customer_id").isNotNull()
)
api_silver = api_silver.filter(
    col("customer_name").isNotNull()
)
api_silver = api_silver.dropDuplicates(
    ["customer_id"]
)


In [0]:
(
    api_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "ingestion_autoloader.ingestion_demo.api_silver"
    )
)


In [0]:
%sql
SELECT *
FROM ingestion_autoloader.ingestion_demo.api_silver;
